# Scalable & Explainable AI-Based Network Intrusion Detection System (NIDS)
**6th-semester internship project | Advanced | Designed for a 16 GB RAM computer**

This notebook demonstrates a reproducible ML workflow: data validation → preprocessing → stratified split → model comparison → metrics → explainability → save model and inference artifacts.

> **Important:** `data/synthetic_nids_sample.csv` is synthetic demonstration data, not real network traffic and not evidence of real-world detection performance. For a credible final project, run the same workflow on CIC-IDS2017 or CSE-CIC-IDS2018 data and report the dataset, split strategy, and results honestly.

## Learning outcomes
- Build a robust baseline using scikit-learn and optionally XGBoost.
- Avoid data leakage by splitting before fitting preprocessing.
- Evaluate with macro-F1, per-class report, confusion matrix, and PR-AUC where applicable.
- Explain predictions with SHAP when installed; otherwise use permutation importance.
- Export a pipeline for use by a future FastAPI/Streamlit application.

## Suggested roadmap
1. **Week 1:** Download one CIC-IDS2017 CSV; inspect schema, labels, duplicates, nulls, and class imbalance.
2. **Week 2:** Clean column names, infinities, missing values; define binary or multiclass target.
3. **Week 3:** Train baseline Random Forest and optional XGBoost; compare macro-F1 and minority-class recall.
4. **Week 4:** Explain predictions with SHAP/permutation importance; inspect false negatives.
5. **Week 5:** Add FastAPI prediction endpoint and Streamlit monitoring dashboard.
6. **Week 6:** Dockerize, document limitations, record reproducible metrics, and prepare resume/demo.

**16 GB RAM guidance:** begin with a 50k–200k row sample, use `float32` for numeric features, avoid loading multiple full CSVs at once, and increase the sample only after profiling memory.


## 1. Install dependencies
Run this cell in Jupyter. If installation fails, run the command in a terminal using the same Python environment as your notebook.

In [ ]:
%pip install -U pandas numpy scikit-learn matplotlib joblib xgboost shap

## 2. Imports and configuration

In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    accuracy_score, f1_score, balanced_accuracy_score
)
from sklearn.inspection import permutation_importance

RANDOM_STATE = 42
DATA_PATH = Path("../data/synthetic_nids_sample.csv")
ARTIFACT_DIR = Path("../artifacts")
ARTIFACT_DIR.mkdir(exist_ok=True)


## 3. Load data and validate

In [ ]:
# The bundled sample is synthetic. Replace DATA_PATH with your downloaded CIC CSV when ready.
df = pd.read_csv(DATA_PATH, low_memory=False)
print("Shape:", df.shape)
display(df.head())
display(df["Label"].value_counts(dropna=False).to_frame("rows"))


In [ ]:
# Standardize column names lightly and clean non-finite values.
df.columns = [str(c).strip() for c in df.columns]
TARGET = "Label"
if TARGET not in df.columns:
    raise ValueError(f"Target column '{TARGET}' not found. Set TARGET to the label column in your dataset.")

df = df.replace([np.inf, -np.inf], np.nan)
df = df.dropna(subset=[TARGET]).drop_duplicates().reset_index(drop=True)

# Keep numeric features only for this starter workflow.
X = df.drop(columns=[TARGET]).select_dtypes(include=[np.number]).copy()
y_raw = df[TARGET].astype(str).str.strip()
if X.shape[1] == 0:
    raise ValueError("No numeric feature columns found.")

# Convert numeric columns to compact float32 where possible.
for col in X.columns:
    X[col] = pd.to_numeric(X[col], errors="coerce").astype("float32")

# Remove all-empty feature columns.
X = X.dropna(axis=1, how="all")
print("Clean shape:", X.shape, "Classes:", y_raw.nunique())
print("Missing-value rate (top 10):")
display(X.isna().mean().sort_values(ascending=False).head(10).to_frame("missing_fraction"))


## 4. Split before fitting any learned preprocessing
For production research, use time-based or capture/day-based splits if records are temporally related. A random split can overestimate performance when similar flows occur in both sets.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y_raw, test_size=0.20, random_state=RANDOM_STATE, stratify=y_raw
)
print("Train:", X_train.shape, "Test:", X_test.shape)
print("Train label proportions:")
display(y_train.value_counts(normalize=True).to_frame("fraction"))


## 5. Train baseline models
Random Forest is the dependable baseline. XGBoost is optional and may take longer; if unavailable, the notebook continues with Random Forest.

In [ ]:
preprocessor = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

rf = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=250, max_depth=None, min_samples_leaf=2,
        class_weight="balanced_subsample", n_jobs=-1,
        random_state=RANDOM_STATE
    ))
])

models = {"Random Forest": rf}
try:
    from xgboost import XGBClassifier
    # XGBoost uses encoded class IDs for multiclass classification.
    label_encoder = LabelEncoder()
    y_train_enc = label_encoder.fit_transform(y_train)
    y_test_enc = label_encoder.transform(y_test)
    xgb = Pipeline(steps=[
        ("preprocess", SimpleImputer(strategy="median")),
        ("model", XGBClassifier(
            n_estimators=300, max_depth=7, learning_rate=0.08,
            subsample=0.85, colsample_bytree=0.85,
            objective="multi:softprob", num_class=len(label_encoder.classes_),
            eval_metric="mlogloss", tree_method="hist",
            n_jobs=4, random_state=RANDOM_STATE
        ))
    ])
    models["XGBoost"] = xgb
except Exception as e:
    print("XGBoost unavailable; continuing with Random Forest.", repr(e))

results = []
fitted_models = {}
for name, model in models.items():
    if name == "XGBoost":
        model.fit(X_train, y_train_enc)
        pred_ids = model.predict(X_test).astype(int)
        pred = label_encoder.inverse_transform(pred_ids)
    else:
        model.fit(X_train, y_train)
        pred = model.predict(X_test)
    fitted_models[name] = model
    results.append({
        "model": name,
        "accuracy": accuracy_score(y_test, pred),
        "balanced_accuracy": balanced_accuracy_score(y_test, pred),
        "macro_f1": f1_score(y_test, pred, average="macro")
    })
pd.DataFrame(results).sort_values("macro_f1", ascending=False)


## 6. Evaluate the selected model
Choose by macro-F1 and per-class recall, not accuracy alone. In intrusion detection, false negatives and class imbalance matter.

In [ ]:
best_name = max(results, key=lambda row: row["macro_f1"])["model"]
best_model = fitted_models[best_name]
if best_name == "XGBoost":
    best_pred = label_encoder.inverse_transform(best_model.predict(X_test).astype(int))
else:
    best_pred = best_model.predict(X_test)

print("Selected model:", best_name)
print(classification_report(y_test, best_pred, zero_division=0))
labels = sorted(y_raw.unique())
cm = confusion_matrix(y_test, best_pred, labels=labels)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
fig, ax = plt.subplots(figsize=(9, 7))
disp.plot(ax=ax, xticks_rotation=45, colorbar=False)
plt.title(f"Confusion Matrix — {best_name}")
plt.tight_layout()
plt.show()


## 7. Explainability
SHAP is preferred for local explanations. For a model-agnostic global fallback, use permutation importance. Explanations describe model behavior, not proof that a flow is malicious.

In [ ]:
# Permutation importance is a reliable fallback and works with the fitted pipeline.
# Use a bounded sample to keep runtime manageable on 16 GB RAM.
sample_n = min(1000, len(X_test))
X_explain = X_test.sample(sample_n, random_state=RANDOM_STATE)
y_explain = y_test.loc[X_explain.index]
perm = permutation_importance(
    best_model, X_explain, y_explain, n_repeats=3,
    random_state=RANDOM_STATE, scoring="f1_macro", n_jobs=2
)
importance = pd.Series(perm.importances_mean, index=X.columns).sort_values(ascending=False)
display(importance.head(15).to_frame("permutation_importance"))
importance.head(15).sort_values().plot(kind="barh", figsize=(8, 6))
plt.title("Global feature importance (permutation)")
plt.tight_layout()
plt.show()


In [ ]:
# Optional SHAP sample explanation. If this errors due to library/model compatibility,
# use the permutation importance results above and document the limitation.
try:
    import shap
    # Explain a small subset only; larger SHAP runs can consume significant memory/time.
    X_shap = X_test.sample(min(200, len(X_test)), random_state=RANDOM_STATE)
    if best_name == "Random Forest":
        transformed = best_model.named_steps["preprocess"].transform(X_shap)
        explainer = shap.TreeExplainer(best_model.named_steps["model"])
        shap_values = explainer.shap_values(transformed)
        print("SHAP computed. For multiclass models, shap_values may be a list/array by class.")
        shap.summary_plot(shap_values, transformed, feature_names=list(X.columns), show=True)
    else:
        print("For this XGBoost pipeline, run SHAP on the transformed feature matrix and the fitted XGBoost estimator.")
except Exception as e:
    print("SHAP step skipped; use permutation importance above. Reason:", repr(e))


## 8. Save model and metadata
The exported artifact is for this notebook's current feature schema. Validate incoming API requests against the same feature names and order.

In [ ]:
artifact_path = ARTIFACT_DIR / "nids_model.joblib"
joblib.dump({
    "model_name": best_name,
    "model": best_model,
    "feature_columns": list(X.columns),
    "target_name": TARGET,
    "classes": sorted(y_raw.unique()),
    "data_note": "Starter notebook was run on synthetic sample unless DATA_PATH was changed."
}, artifact_path)
print("Saved:", artifact_path.resolve())


## 9. Predict one new record
Enter values in the same feature columns used during training. This is a demonstration only; a real deployment must validate ranges, handle unknown labels, log decisions securely, and be tested on unseen capture periods.

In [ ]:
# Example: use one test row as a schema-valid demo input.
demo = X_test.iloc[[0]].copy()
demo_prediction = best_model.predict(demo)
print("Predicted class:", demo_prediction[0])
display(demo)


## 10. How to use real CIC data
1. Download a flow CSV from the official CIC-IDS2017 page.
2. Copy it to `data/` and set `DATA_PATH` to its filename.
3. Set `TARGET` to the actual label column (often ` Label` or `Label`); inspect exact column names first.
4. Do not combine unrelated CSVs without reconciling schemas and labels.
5. For 16 GB RAM, load a single CSV, select numeric columns, and optionally use `pd.read_csv(..., nrows=200_000)` for a first experiment.
6. Report results only from the real dataset run; keep the synthetic demo clearly identified.
7. For stronger evaluation, split by capture day/attack scenario instead of random rows, and consider binary detection before multiclass classification.

## Resume bullet (use after you have actually run and verified the project)
**Scalable & Explainable AI-Based Network Intrusion Detection System** — Built a Python-based intrusion-detection ML pipeline using Random Forest/XGBoost, engineered flow-level features, evaluated models with macro-F1 and per-class recall, and generated model explanations using SHAP/permutation importance. Packaged the trained pipeline for future API/dashboard integration.

## Limitations and responsible use
- Synthetic data is only for verifying the code path; it cannot establish real detection performance.
- Public IDS datasets may contain artifacts and distribution shifts; validate on a separate capture scenario.
- This is a defensive research project. Do not deploy blocking actions based only on model predictions without human review and security testing.
